101 test

In [ ]:
import requests

url = "http://aisdata.ais.dk/2024/aisdk-2024-03-01.zip"

print("Trying to download:", url)

r = requests.get(url)

print("Status code:", r.status_code) 
print("File size (MB):", len(r.content) / 1e6) # Print file size in megabytes

Trying to download: http://aisdata.ais.dk/2024/aisdk-2024-03-01.zip
Status code: 200
File size (MB): 576.627852


lad os prøve at åbne zip-filen uden at gemme den og istedet have den direkte i memory

In [3]:
import zipfile
import io

z = zipfile.ZipFile(io.BytesIO(r.content))

print("Files inside zip:")
print(z.namelist())

Files inside zip:
['aisdk-2024-03-01.csv']


In [4]:
import pandas as pd

csv_name = z.namelist()[0]

with z.open(csv_name) as f:

    for chunk in pd.read_csv(f, chunksize=100000):

        print("Chunk rows:", len(chunk))
        break

Chunk rows: 100000


In [5]:
print(chunk.columns)

Index(['# Timestamp', 'Type of mobile', 'MMSI', 'Latitude', 'Longitude',
       'Navigational status', 'ROT', 'SOG', 'COG', 'Heading', 'IMO',
       'Callsign', 'Name', 'Ship type', 'Cargo type', 'Width', 'Length',
       'Type of position fixing device', 'Draught', 'Destination', 'ETA',
       'Data source type', 'A', 'B', 'C', 'D'],
      dtype='object')


In [13]:
print(chunk['Ship type'].unique())
print(chunk['Ship type'].value_counts())
print(chunk['Ship type'].nunique())

['Undefined' 'Tanker' 'Passenger' 'Pleasure' 'Sailing' 'Reserved' 'Cargo'
 'Fishing' 'Towing' 'Diving' 'Military' 'Pilot' 'HSC' 'Anti-pollution'
 'SAR' 'Dredging' 'Other' 'Tug' 'Port tender' 'Law enforcement'
 'Not party to conflict' 'Medical' 'Spare 1' 'Towing long/wide']
Ship type
Undefined                42823
Cargo                    14195
Fishing                  14170
Passenger                 7332
Tanker                    6567
Other                     3224
Tug                       2088
Pilot                     2068
Dredging                  1935
SAR                       1520
Pleasure                   818
HSC                        673
Sailing                    636
Military                   628
Reserved                   412
Law enforcement            255
Towing                     234
Port tender                189
Spare 1                     89
Not party to conflict       65
Medical                     52
Diving                      13
Anti-pollution               8
Tow

In [17]:
chunk.tail(25)

,# Timestamp,Type of mobile,MMSI,Latitude,Longitude,Navigational status,ROT,SOG,COG,Heading,...,Length,Type of position fixing device,Draught,Destination,ETA,Data source type,A,B,C,D
99975,01/03/2024 00:09:57,Class A,219024177,56.094540,12.457918,Under way using engine,0.0,0.0,258.5,244.0,...,15.0,GPS,1.5,Unknown,NaN,AIS,12.0,3.0,2.0,3.0
99976,01/03/2024 00:09:57,Class A,219836000,57.200817,11.618417,Under way using engine,-1.1,18.9,342.1,342.0,...,399.0,GPS,12.4,DKAAR>>DEBRV,01/03/2024 22:00:00,AIS,116.0,283.0,29.0,29.0
99977,01/03/2024 00:09:57,Class A,219836000,57.200817,11.618417,Under way using engine,-1.1,18.9,342.1,342.0,...,399.0,GPS,12.4,DKAAR>>DEBRV,01/03/2024 22:00:00,AIS,116.0,283.0,29.0,29.0
99978,01/03/2024 00:09:57,Class A,219836000,57.200817,11.618417,Under way using engine,-1.1,18.9,342.1,342.0,...,399.0,GPS,12.4,DKAAR>>DEBRV,01/03/2024 22:00:00,AIS,116.0,283.0,29.0,29.0
99979,01/03/2024 00:09:57,Class A,218098000,57.923322,9.286367,Under way using engine,0.0,13.5,27.2,30.0,...,149.0,GPS,7.8,NO MSS,01/03/2024 06:30:00,AIS,137.0,12.0,13.0,10.0
99980,01/03/2024 00:09:57,Class A,218098000,57.923322,9.286367,Under way using engine,0.0,13.5,27.2,30.0,...,149.0,GPS,7.8,NO MSS,01/03/2024 06:30:00,AIS,137.0,12.0,13.0,10.0
99981,01/03/2024 00:09:57,Class A,219836000,57.200817,11.618417,Under way using engine,-1.1,18.9,342.1,342.0,...,399.0,GPS,12.4,DKAAR>>DEBRV,01/03/2024 22:00:00,AIS,116.0,283.0,29.0,29.0
99982,01/03/2024 00:09:57,Class A,636015378,55.480588,5.117675,Under way sailing,NaN,0.0,210.3,NaN,...,80.0,GPS,10.0,DAN E,NaN,AIS,70.0,10.0,45.0,25.0
99983,01/03/2024 00:09:57,Class A,354294000,55.545580,15.011397,Under way using engine,0.0,18.6,245.6,246.0,...,143.0,GPS,6.9,SE GOT,01/03/2024 23:00:00,AIS,118.0,25.0,15.0,7.0
99984,01/03/2024 00:09:57,Class A,354294000,55.545580,15.011397,Under way using engine,0.0,18.6,245.6,246.0,...,143.0,GPS,6.9,SE GOT,01/03/2024 23:00:00,AIS,118.0,25.0,15.0,7.0


### DATA SCRAPING -> IMO-numbers on Tankers in perioede between 2022-2025

In [ ]:
import requests
import pandas as pd
import zipfile
import io
from datetime import date, timedelta

all_imos = set()

start = date(2022,1,1)
end = date(2025,12,31)

d = start

while d <= end:

    url = f"http://aisdata.ais.dk/{d.year}/aisdk-{d}.zip"

    print("Processing:", url)

    try:

        r = requests.get(url)

        z = zipfile.ZipFile(io.BytesIO(r.content))

        csv_name = z.namelist()[0]

        with z.open(csv_name) as f:

            for chunk in pd.read_csv(f, chunksize=100000):

                # find tankers
                tankers = chunk[chunk["Ship type"] == "Tanker"]

                # get IMO numbers
                imos = tankers["IMO"].dropna()

                # add to set (keeps only unique)
                all_imos.update(imos)

    except Exception as e:

        print("Could not process:", url)

    d += timedelta(days=1)

print("Unique tanker IMOs:", len(all_imos))

Data scrapier with auto-saver and resume/continue after interruption

In [18]:
import requests
import pandas as pd
import zipfile
import io
from datetime import date, timedelta
from pathlib import Path

output_file = Path("tanker_imos.txt")

# load existing IMOs if file already exists
if output_file.exists():

    print("Loading existing IMOs...")

    with open(output_file) as f:
        all_imos = set(line.strip() for line in f)

else:
    all_imos = set()

print("Starting with", len(all_imos), "known IMOs")


start = date(2022,1,1)
end = date(2025,12,31)

d = start

while d <= end:

    url = f"http://aisdata.ais.dk/{d.year}/aisdk-{d}.zip"

    print("Processing:", url)

    try:

        r = requests.get(url)

        z = zipfile.ZipFile(io.BytesIO(r.content))

        csv_name = z.namelist()[0]

        with z.open(csv_name) as f:

            for chunk in pd.read_csv(f, chunksize=100000):

                tankers = chunk[chunk["Ship type"] == "Tanker"]

                imos = tankers["IMO"].dropna()

                all_imos.update(str(i) for i in imos)

        # save progress after each day
        with open(output_file, "w") as f:
            for imo in sorted(all_imos):
                f.write(imo + "\n")

        print("Saved progress. Unique IMOs:", len(all_imos))

    except Exception:

        print("Skipping:", url)

    d += timedelta(days=1)

print("FINAL UNIQUE IMOs:", len(all_imos))

Starting with 0 known IMOs
Processing: http://aisdata.ais.dk/2022/aisdk-2022-01-01.zip
Skipping: http://aisdata.ais.dk/2022/aisdk-2022-01-01.zip
Processing: http://aisdata.ais.dk/2022/aisdk-2022-01-02.zip
Skipping: http://aisdata.ais.dk/2022/aisdk-2022-01-02.zip
Processing: http://aisdata.ais.dk/2022/aisdk-2022-01-03.zip
Skipping: http://aisdata.ais.dk/2022/aisdk-2022-01-03.zip
Processing: http://aisdata.ais.dk/2022/aisdk-2022-01-04.zip
Skipping: http://aisdata.ais.dk/2022/aisdk-2022-01-04.zip
Processing: http://aisdata.ais.dk/2022/aisdk-2022-01-05.zip
Skipping: http://aisdata.ais.dk/2022/aisdk-2022-01-05.zip
Processing: http://aisdata.ais.dk/2022/aisdk-2022-01-06.zip
Skipping: http://aisdata.ais.dk/2022/aisdk-2022-01-06.zip
Processing: http://aisdata.ais.dk/2022/aisdk-2022-01-07.zip
Skipping: http://aisdata.ais.dk/2022/aisdk-2022-01-07.zip
Processing: http://aisdata.ais.dk/2022/aisdk-2022-01-08.zip
Skipping: http://aisdata.ais.dk/2022/aisdk-2022-01-08.zip
Processing: http://aisdata.ai

KeyboardInterrupt: 